# SimJEB 2 - training

Trains the surface-stress surrogate on the cached graphs from notebook 1.

**Settings required:** Accelerator **GPU T4**, Internet **ON** (for the pip install).
Add notebook 1's output as an input dataset. Run with *Save & Run All (Commit)*.

### This notebook is meant to be run more than once

At roughly 20-60 s per epoch, a 3,000-epoch budget does not fit a 12-hour session. So
it stops cleanly at a wall-clock limit and writes a full checkpoint every epoch --
model, optimiser, scheduler, history. **Re-running continues from where it stopped.**

To continue: add the *previous run's output* as a second input dataset and set
`RESUME_FROM` below. Early stopping may well fire first.

In [ ]:
GITHUB_REPO = "https://github.com/Vedavamsi-3/simjeb-structural-gnn.git"          # same as notebook 1
DATA        = "/kaggle/input/simjeb-data"   # <- notebook 1's output dataset
RESUME_FROM = ""          # <- a previous run's output, to continue a long job

REPO = "/kaggle/working/simjeb-structural-gnn"

import subprocess, sys, os, shutil
from pathlib import Path

if GITHUB_REPO and not Path(REPO).exists():
    subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO, REPO], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch-geometric", "meshio", "trimesh"], check=True)

sys.path.insert(0, REPO)
os.chdir(REPO)

import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
# Kaggle mounts input datasets under a name it chooses, so rather than hard-coding
# a path, find whichever mounted dataset actually contains the built graphs.
# Kaggle mounts inputs at a path it chooses, and the depth varies -- sometimes
# /kaggle/input/<name>/graphs, sometimes /kaggle/input/datasets/<name>/graphs. Search
# a few levels rather than assuming one.
_found = [p for p in Path("/kaggle/input").rglob("graphs") if p.is_dir()]
if _found:
    DATA = str(_found[0].parent)
    print("found data at:", DATA)
else:
    print("no graphs/ found under /kaggle/input -- did you Add Data?")
    for p in Path("/kaggle/input").rglob("*"):
        if p.is_dir() and len(p.relative_to("/kaggle/input").parts) <= 2:
            print("   ", p)
_resume = sorted(Path("/kaggle/input").glob("*/outputs/C/checkpoint.pt"))
if _resume and not RESUME_FROM:
    RESUME_FROM = str(_resume[0].parents[2])
    print("found a previous run to resume:", RESUME_FROM)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

OUT = Path("/kaggle/working/outputs")
RUN = "C"

# Copy a previous run's checkpoint into place -- Kaggle input datasets are read-only,
# and the loop needs to write to the same directory it reads from.
if RESUME_FROM:
    previous = Path(RESUME_FROM) / "outputs" / RUN
    if previous.exists():
        (OUT / RUN).mkdir(parents=True, exist_ok=True)
        for name in ("checkpoint.pt", "best_model.pt", "history.csv"):
            if (previous / name).exists():
                shutil.copy(previous / name, OUT / RUN / name)
        print("resuming from", previous)
    else:
        print("RESUME_FROM set but not found:", previous)

## Configuration

The choices that are not defaults, and why:

| setting | value | reason |
|---|---|---|
| `use_material` | **on** | constant across all 381 models so it cannot help, but it costs three columns and the pipeline then extends unchanged to a multi-material dataset |
| `use_position` | **off** | the brackets share a frame so absolute coordinates are meaningful, but with ~274 training shapes they are also easy to memorise; `dist_to_clamp` and `dist_to_load` carry the useful part in a form that transfers |
| `use_aux_displacement` | **on**, weight 0.2 | stress is the hard, gradient-derived quantity; supervising the easy one forces the shared trunk to learn the deformation mechanics stress depends on. Discarded at inference |
| `hidden_dim` / `num_blocks` | **64 / 8** | not the paper's 128/15. Depth is set by the length scale of stress concentration (~8 hops = 7.6 mm, covering fillet radii), not by the load path -- which is 54 hops away and would need 10.6 GB per graph |
| `log_stress` | **on** | peak von Mises reaches 17x the 880 MPa yield at singular corners; without the log a handful of nodes supply most of the gradient |
| `max_epochs` | **3000** | a ceiling, not a target -- early stopping will likely fire first |

In [ ]:
from src.train import TrainConfig, train

config = TrainConfig(
    run_name=RUN,
    graph_dir=f"{DATA}/graphs",
    split_path=f"{DATA}/splits/grouped_split_v1.json",
    out_dir=str(OUT),
    load_case="ver",

    use_material=True,
    use_position=False,
    use_aux_displacement=True,
    aux_weight=0.2,
    log_stress=True,

    hidden_dim=64,
    num_blocks=8,

    batch_size=2,            # graphs range 9k to 309k nodes
    use_checkpointing=True,  # the largest needs ~17 GB without it
    lr=1e-3,
    weight_decay=1e-5,
    max_epochs=3000,
    patience=200,
    max_hours=10.5,       # inside a 12 h session, with margin to write the checkpoint

    seed=0,
    device="cuda",
    amp=True,
    num_workers=2,
    in_memory=True,       # ~274 graphs of ~6 MB; removes disk reads from the epoch loop
)

for key in ("graph_dir", "split_path"):
    assert Path(getattr(config, key)).exists(), f"{key} not found: {getattr(config, key)}"
print("inputs found")

## Timing check

Run a handful of epochs first and read the real epoch time off the log. That decides
whether 3,000 epochs is one session or four -- and it is much cheaper to learn now
than after ten hours.

In [ ]:
import dataclasses, time

probe = dataclasses.replace(config, run_name="timing_probe", max_epochs=3,
                            patience=10_000, max_hours=0.5)
began = time.time()
probe_result = train(probe)
per_epoch = (time.time() - began) / max(len(probe_result.history), 1)

print(f"\n~{per_epoch:.1f} s per epoch")
print(f"  {config.max_hours} h  -> ~{int(config.max_hours*3600/per_epoch)} epochs per session")
print(f"  {config.max_epochs} epochs -> ~{config.max_epochs*per_epoch/3600:.1f} h total")
shutil.rmtree(OUT / "timing_probe", ignore_errors=True)

## Train

In [ ]:
%%time
result = train(config)

In [ ]:
import pandas as pd
from IPython.display import Image, display

history = pd.read_csv(OUT / RUN / "history.csv")
print(result.stopped_because)
print(f"epochs completed : {len(history)}")
print(f"best epoch       : {result.best_epoch}")
print(f"best val loss    : {result.best_val_loss:.5f}")
print(f"best val R2      : {history.val_r2_mpa.max():.4f}")
print(f"best val MAE     : {history.val_mae_mpa.min():.1f} MPa")
display(Image(str(OUT / RUN / f"loss_curve_{RUN}.png")))

### Reading the curves

- **Validation still falling at the end** -- it stopped on the clock, not on
  convergence. Re-run this notebook with `RESUME_FROM` set to this run's output.
- **Validation flat while training falls** -- overfitting. More weight decay, or a
  smaller model.
- **Both flat and high** -- underfitting. More capacity, or a higher learning rate.
- **Both fallen and levelled together** -- converged. Move to notebook 3.

In [ ]:
final = history.tail(min(50, len(history)))
improving = final.val_loss.iloc[-1] < final.val_loss.iloc[0]
gap = history.val_loss.iloc[-1] / max(history.train_loss.iloc[-1], 1e-12)

print(f"validation still improving over the last {len(final)} epochs: {improving}")
print(f"val/train loss ratio: {gap:.2f}")
if "wall-clock" in result.stopped_because:
    print("\n-> stopped on time, not convergence. Re-run with RESUME_FROM set to this output.")
elif improving:
    print("\n-> early stopping fired but validation was still drifting down; consider more patience.")
else:
    print("\n-> converged. Go to notebook 3.")